In [ ]:
# GPU preflight check — run this before launching vLLM.
# Shows which GPU was assigned, VRAM tier, and recommended flags.
# Exits with a clear error if the node is incompatible.
import subprocess, sys
from pathlib import Path

script = Path.home() / "llm_train" / "tools" / "start_vllm.py"
if not script.exists():
    raise FileNotFoundError(f"start_vllm.py not found at {script} — is the repo cloned to ~/llm_train?")

result = subprocess.run(
    [sys.executable, str(script), "--dry-run"],
    capture_output=False,  # prints directly to notebook output
)
if result.returncode != 0:
    print("\n[ACTION REQUIRED] Preflight failed. Request a different GPU node on NRP and restart your pod.")


## vLLM Startup

Run the preflight cell below first, then copy the terminal command to launch the server.

For GPU compatibility check + adaptive launch (recommended):
```bash
cd ~/llm_train
python tools/start_vllm.py
```

Optional overrides:
```bash
python tools/start_vllm.py --adapter-path checkpoints/my_other_run/adapter_model
python tools/start_vllm.py --dry-run   # print command without launching
```

Monitor / verify:
```bash
tail -f ~/llm_train/vllm.log
curl -s http://127.0.0.1:8000/v1/models
```

---

**Manual launch (fallback reference — parameters fixed, no preflight)**:
```bash
pkill -f vllm.entrypoints.openai.api_server || true
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

nohup env \
  HF_HOME=/home/jovyan/llm_train/.hf_home \
  HF_HUB_CACHE=/home/jovyan/llm_train/.hf_home/hub \
  TRANSFORMERS_CACHE=/home/jovyan/llm_train/.hf_home/hub \
  python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-VL-7B-Instruct \
  --download-dir /home/jovyan/llm_train/.hf_home/hub \
  --host 127.0.0.1 \
  --port 8000 \
  --trust-remote-code \
  --enable-lora \
  --max-loras 1 \
  --max-lora-rank 64 \
  --lora-modules spine_adapter=/home/jovyan/llm_train/checkpoints/qwen2_5_vl_lora_512Res/adapter_model \
  --dtype float16 \
  --max-model-len 2048 \
  --max-num-seqs 1 \
  --gpu-memory-utilization 0.82 \
  --enforce-eager \
  --limit-mm-per-prompt '{"image": 1, "video": 0}' > vllm.log 2>&1 &
```

In [ ]:
#non image test
import requests

response = requests.post(
    "http://127.0.0.1:8000/v1/completions",
    json={
        "model": "spine_adapter",
        "prompt": "Say hello from the LoRA adapter test.",
        "max_tokens": 32,
    },
    timeout=60,
)

print(response.status_code)
print(response.text[:1000])

In [ ]:
# image test

import base64
import json
import mimetypes
from pathlib import Path

import requests

image_path = Path("/home/jovyan/llm_train/img/0aba1c3b-IMG_3566_spine_028.png")
if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path}")

mime_type, _ = mimetypes.guess_type(str(image_path))
mime_type = mime_type or "image/png"

image_b64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")
image_data_url = f"data:{mime_type};base64,{image_b64}"

payload = {
    "model": "spine_adapter",
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Extract the book information from this spine image. Return strict JSON with keys: title, author, call_no."
                },
                {
                    "type": "image_url",
                    "image_url": {"url": image_data_url}
                }
            ]
        }
    ],
    "max_tokens": 256,
    "temperature": 0.0
}

resp = requests.post(
    "http://127.0.0.1:8000/v1/chat/completions",
    json=payload,
    timeout=300,
)

print("status:", resp.status_code)
resp.raise_for_status()

data = resp.json()
print(json.dumps(data, indent=2)[:4000])

if data.get("choices"):
    print("\nModel text output:\n")
    print(data["choices"][0]["message"]["content"])